# **Testing REST API**

## **Import Library & Definisi URL**

In [ ]:
import os
import json
import base64
import requests
import tensorflow as tf
from dotenv import load_dotenv

# Load variabel dari .env
load_dotenv()

# Ambil URL dari .env
RAILWAY_URL = os.getenv("RAILWAY_URL")

if not RAILWAY_URL:
    raise ValueError(
        "RAILWAY_URL tidak ditemukan. Pastikan kamu sudah membuat file .env "
        "dan mengisi variabelnya dengan benar."
    )

MODEL_METADATA_URL = f"{RAILWAY_URL}/v1/models/adult-income-model/metadata"
MODEL_PREDICT_URL = f"{RAILWAY_URL}/v1/models/adult-income-model:predict"

## **Uji Status & Metadata Model**

In [ ]:
response = requests.get(MODEL_METADATA_URL)

print("GET Request Status Code:", response.status_code)
print("\nRespon Metadata Model:")
print(json.dumps(response.json(), indent=2))

Pada tahap ini, kita melakukan panggilan HTTP GET ke endpoint `/metadata` untuk memverifikasi bahwa:

- Server TensorFlow Serving di Railway berjalan aktif (Status Code: 200 OK).
- Signature definition `serving_default` telah terkonfigurasi dengan benar untuk menerima input berformat `tf.train.Example` ter-serialisasi.

## **Uji Inference**

In [ ]:
def create_tf_example(data):
    """Mengonversi dictionary data mentah menjadi serialized TF Example string."""
    feature = {}
    for key, val in data.items():
        if isinstance(val, float):
            feature[key] = tf.train.Feature(float_list=tf.train.FloatList(value=[val]))
        elif isinstance(val, int):
            feature[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=[val]))
        elif isinstance(val, str):
            feature[key] = tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[val.encode("utf-8")])
            )

    example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
    return example_proto.SerializeToString()


# Data tes individu sampel
sample_person = {
    "age": 39,
    "workclass": "State-gov",
    "fnlwgt": 77516,
    "education": "Bachelors",
    "education.num": 13,
    "marital.status": "Never-married",
    "occupation": "Adm-clerical",
    "relationship": "Not-in-family",
    "race": "White",
    "sex": "Male",
    "capital.gain": 2174,
    "capital.loss": 0,
    "hours.per.week": 40,
    "native.country": "United-States",
}

# Serialisasi dan Encode Base64
serialized_example = create_tf_example(sample_person)
b64_example = base64.b64encode(serialized_example).decode("utf-8")

payload = {
    "signature_name": "serving_default",
    "instances": [{"b64": b64_example}],
}

# Kirim POST Request ke Railway
response = requests.post(MODEL_PREDICT_URL, json=payload)
result = response.json()

print("POST Request Status Code:", response.status_code)
print("Respon Prediksi Raw:", result)

# Interpretasi hasil prediksi
prediction_score = result["predictions"][0][0]
print(f"\nSkor Probabilitas Pendapatan >50K: {prediction_score:.4f}")

if prediction_score >= 0.5:
    print("Hasil Prediksi: Pendapatan Tinggi (>50K/tahun)")
else:
    print("Hasil Prediksi: Pendapatan Standar (<=50K/tahun)")

Pada tahap ini, kita mengirimkan data individu baru ke cloud server via HTTP POST request:

1. **Serialisasi Data:** Mengubah dictionary data individu menjadi protobuf `tf.train.Example` dan di-encode ke format Base64.
2. **Inference Request:** Mengirimkan payload JSON berisi data Base64 ke endpoint `:predict`.
3. **Interpretasi Output:** Menerima skor probabilitas (0.0 – 1.0) dan menentukan klasifikasi pendapatan (0: <=50K, 1: >50K).